#### Libraries

In [69]:
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import random
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint

# Plotting style
sns.set_style('darkgrid')
sns.set_theme(font_scale=1.)

# Set the state (we will call the function when we need to make sure that we get the same results every time we run the code)
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_all_seeds(1234)

In [ ]:
def get_basic_model(input_dim, hidden_dim, output_dim):
    # Assignment: Define the model usign the torch.nn.Sequential approach (see the exercise part for help)
    # YOUR CODE HERE
    return torch.nn.Sequential(
        torch.nn.Linear(in_features=input_dim, out_features=hidden_dim, bias=True),     # Input layer
        torch.nn.ReLU(),                                                                # Activation function
        torch.nn.Linear(in_features=hidden_dim, out_features=output_dim, bias=True),    # Output layer
    ) 


In [ ]:
def fit_nn(model, X_train, y_train, X_test, y_test, criterion,n_epochs,lr):
    # Assignment: Define the 'optimizer' as stochastic gradient descent (SGD) method with the learning rate defined in lr.     
    optimizer = torch.optim.SGD(params=model.parameters(), lr=lr)
    # YOUR CODE HERE

    # Define a dictionary to store the loss values for each epoch
    results = {'train': [], 'test': []}

    # Training loop
    for epoch in range(n_epochs):    
        # Set the model to training mode
        model.train()
        # Make sure that the gradients are zero before you use backpropagation
        optimizer.zero_grad()
        # Make a forward pass through the model to compute the outputs
        outputs = model(X_train)
        # Compute the loss
        loss = criterion(outputs, y_train)
        # Do a backward pass to compute the gradients wrt. model parameters using backpropagation.
        loss.backward()
        # Update the model parameters by making the optimizer take a gradient descent step
        optimizer.step()

        with torch.no_grad(): # No need to compute gradients for the validation set
            # Set the model to evaluation mode
            model.eval()  
            # Compute the loss for the test set
            test_outputs = model(X_test)
            test_loss = criterion(test_outputs, y_test)
        
        # Store the training and test loss for this epoch in the dictionary
        results['train'].append(loss.item())
        results['test'].append(test_loss.item())

        # Print the loss every 1000 epochs
        if (epoch+1) % 1000 == 0:
            print(f'Epoch [{epoch+1}/{n_epochs}], Training loss: {loss.item():.4f}, Test Loss: {test_loss.item():.4f}')

    return model, results

In [ ]:
df = pd.read_csv('HR_data.csv')

print(df['Phase'].dtype, df['Phase'].unique())

hr_cols = ['HR_Mean', 'HR_Median', 'HR_Min', 'HR_Max', 'HR_AUC']
group_cols = ['Round', 'Individual']   

baseline = df[df['Phase'] == 'phase1'].set_index(group_cols)[hr_cols]
baseline = baseline.rename(columns={c: c + '_baseline' for c in hr_cols})

df = df[df['Phase'].isin(['phase2', 'phase3'])].copy()
df = df.merge(baseline, on=group_cols, how='left')

for c in hr_cols:
    df[c] = df[c] - df[c + '_baseline']

df = df.drop(columns=[c + '_baseline' for c in hr_cols])


X = df.drop(columns=['Cohort', 'Round', 'Frustrated', 'HR_AUC', 'HR_std', 'HR_Median'])
X = X.iloc[:, 1:]
X['Phase'] = X['Phase'].map({'phase1': 1, 'phase2': 2, 'phase3': 3})
y = df['Frustrated']

print(X.shape)
X

In [ ]:
groups = df['Individual']

X_np = X.drop(columns=['Individual']).values.astype(np.float32)
y_np = y.values.astype(np.float32)   

gkf = GroupKFold(n_splits=7)

n_epochs_list = [10, 50, 100, 200, 1000]
lr_list = [0.1, 0.01, 0.001, 0.0001]

table_results = []

for i in n_epochs_list:
    for j in lr_list:
        fold_test_losses = []
        for fold, (train_idx, test_idx) in enumerate(gkf.split(X_np, y_np, groups=groups)):

            scaler = StandardScaler().fit(X_np[train_idx])

            X_train = torch.tensor(scaler.transform(X_np[train_idx]).astype(np.float32))
            X_test  = torch.tensor(scaler.transform(X_np[test_idx]).astype(np.float32))

            y_train = torch.tensor(y_np[train_idx]).unsqueeze(1)
            y_test  = torch.tensor(y_np[test_idx]).unsqueeze(1)

            # Here we call a specific instance of the model
            hidden_dim = 50
            input_dim = X_train.shape[1]
            output_dim = 1

            model = get_basic_model(input_dim, hidden_dim, output_dim)

            criterion = torch.nn.MSELoss()

            model, results = fit_nn(model, X_train, y_train, X_test, y_test, criterion, n_epochs=i, lr=j)

            final_test_loss = results['test'][-1]
            fold_test_losses.append(final_test_loss)

            print(f"epochs={i}, lr={j}, Fold {fold}: final train loss={results['train'][-1]:.4f}, final test loss={final_test_loss:.4f}")
        avg_test_mse = np.mean(fold_test_losses)
        root_avg_test_mse = np.sqrt(np.mean(fold_test_losses))
        std_test_mse = np.std(fold_test_losses)

        table_results.append({
            'n_epochs': i,
            'lr': j,
            'avg_test_mse': avg_test_mse,
            'root_avg_test_mse': root_avg_test_mse,
            'std_test_mse': std_test_mse,
            'fold_test_losses': fold_test_losses  # keep raw per-fold values too, in case you want them later
        })

        print(f"--> epochs={i}, lr={j}: avg test MSE={avg_test_mse:.4f} root avg test MSE={root_avg_test_mse:.4f} (+/- {std_test_mse:.4f})\n")

results_df = pd.DataFrame(table_results)
results_df_sorted = results_df.sort_values('avg_test_mse').reset_index(drop=True)
results_df_sorted


In [61]:
X

,HR_Mean,HR_Min,HR_Max,Phase,Individual,Puzzler
0,4.593227,5.35,3.15,3,1,1
1,-2.390863,-0.76,-2.00,2,1,1
2,4.544762,1.05,5.70,3,1,1
3,2.950165,-0.75,10.93,2,1,1
4,3.362441,4.95,2.05,3,1,1
...,...,...,...,...,...,...
107,6.450080,3.15,-11.88,2,14,0
108,10.549731,1.41,22.19,3,14,0
109,15.754642,4.46,19.39,2,14,0
110,-6.896583,1.80,-41.70,3,14,0


In [72]:
groups = df['Individual']

X_np = X.drop(columns=['Individual']).values.astype(np.float32)
y_np = y.values 

gkf = GroupKFold(n_splits=7)

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_np, y_np, groups=groups)):
    X_train = X_np[train_idx]
    X_test  = X_np[test_idx]

    y_train = y_np[train_idx]
    y_test  = y_np[test_idx]

    rf = RandomForestClassifier()
    rf = rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print("Accuracy:", accuracy)
    

Accuracy: 0.3125
Accuracy: 0.125
Accuracy: 0.25
Accuracy: 0.1875
Accuracy: 0.25
Accuracy: 0.125
Accuracy: 0.1875
